# Lesson 19: Optical Flow and Motion Estimation

Optical flow estimates apparent motion between two frames of a video: for each pixel (or a chosen set of points), a 2D vector $(u, v)$ describing where it moved to. This lesson derives the classic **Lucas-Kanade** method from the same structure tensor used for corner detection in Lesson 18, confronts the fundamental **aperture problem**, and finishes with dense flow via the **Farneback** method.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## The brightness constancy assumption

Optical flow assumes a point's intensity doesn't change as it moves: $I(x, y, t) = I(x+u, y+v, t+1)$. A first-order Taylor expansion of the right side gives the **optical flow constraint equation**:

$$I_x u + I_y v + I_t = 0$$

where $I_x, I_y$ are the spatial gradients (Lesson 12) and $I_t$ is the frame-to-frame intensity difference. This is **one equation with two unknowns** ($u$ and $v$) at every single pixel &mdash; not enough information on its own to solve for the flow.

## The aperture problem

Looking through a small window at a moving edge, only the motion **perpendicular to the edge** is visible; motion **along the edge** produces no visible change at all, and so is invisible to a local measurement. This is exactly why the flow constraint equation is underdetermined: it only ever constrains the component of $(u,v)$ along the gradient direction $(I_x, I_y)$, leaving the perpendicular component completely unconstrained by that one equation.

In [ ]:
edge = np.zeros((100, 100), dtype=np.uint8)
edge[:, 50:] = 200

true_motion = np.array([3, 4])       # moves diagonally: 3 right, 4 down
shift = np.float32([[1, 0, true_motion[0]], [0, 1, true_motion[1]]])
edge_moved = cv2.warpAffine(edge, shift, (100, 100))

# The along-edge component of the true motion (vertical, since the edge is vertical)
# leaves the local appearance near the edge completely unchanged.
along_edge_only = cv2.warpAffine(edge, np.float32([[1, 0, 0], [0, 1, true_motion[1]]]), (100, 100))

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, im, title in zip(axes, [edge, edge_moved, along_edge_only],
                          ['Original edge', 'True motion (3,4)', 'Vertical-only motion (0,4)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

# Compare away from the top/bottom border, where the vertical shift trivially exposes
# blank rows -- that border effect is a finite-image artifact, not the phenomenon we're after.
interior = slice(10, 90)
print('vertical-only motion looks identical to the original (away from the border)?',
      np.array_equal(edge[interior, :], along_edge_only[interior, :]))

Sliding the edge straight down produces *no visible change* &mdash; a purely vertical shift of a vertical edge is completely invisible locally. Only the horizontal component of any motion near this edge can be recovered from local appearance alone.

## Lucas-Kanade: solving the aperture problem with a window

<a href="../references.html#lucas-kanade-1981">Lucas and Kanade's fix (1981)</a>: assume the flow $(u,v)$ is **constant over a small window**, then combine the flow constraint equation from every pixel in that window into an overdetermined least-squares system:

$$\underbrace{\begin{bmatrix}\sum I_x^2 & \sum I_xI_y \\ \sum I_xI_y & \sum I_y^2\end{bmatrix}}_{M}\begin{bmatrix}u\\v\end{bmatrix} = -\begin{bmatrix}\sum I_xI_t\\ \sum I_yI_t\end{bmatrix}$$

$M$ is exactly the structure tensor from Lesson 18! This is not a coincidence: solving this system requires $M$ to be invertible, i.e. to have two large eigenvalues &mdash; precisely the Shi-Tomasi "good feature to track" condition. A flat region ($M$ near zero) or an edge (one small eigenvalue &mdash; the aperture problem again) gives an ill-conditioned or singular system; a corner gives a well-conditioned one. **Corners are trackable for exactly the same reason they're good corners.**

In [ ]:
rng = np.random.default_rng(0)
frame1 = np.zeros((200, 200), dtype=np.uint8)
for _ in range(20):
    x, y = rng.integers(20, 180, 2)
    radius = rng.integers(5, 15)
    cv2.circle(frame1, (x, y), radius, int(rng.integers(100, 255)), -1)

small_motion = (0.6, 0.4)  # sub-pixel motion, well inside the linear (Taylor) approximation's validity
shift = np.float32([[1, 0, small_motion[0]], [0, 1, small_motion[1]]])
frame2_small = cv2.warpAffine(frame1, shift, (200, 200))

Ix = cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3, scale=1 / 8)
Iy = cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3, scale=1 / 8)
It = frame2_small.astype(np.float64) - frame1.astype(np.float64)


def lucas_kanade_at(x, y, half_win=7):
    x, y = int(round(x)), int(round(y))
    ix = Ix[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    iy = Iy[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    it = It[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    A = np.stack([ix, iy], axis=1)
    solution, *_ = np.linalg.lstsq(A, -it, rcond=None)
    return solution

corners = cv2.goodFeaturesToTrack(frame1, maxCorners=30, qualityLevel=0.1, minDistance=10)
flows = np.array([lucas_kanade_at(p[0][0], p[0][1]) for p in corners])

print(f'true motion:        {small_motion}')
print(f'manual LK estimate: ({flows[:, 0].mean():.3f}, {flows[:, 1].mean():.3f})  (averaged over {len(corners)} corners)')

### Larger motions need iteration

This single-shot linear solve relies on the Taylor approximation, which only holds for small motions. For a bigger shift, the same one-shot approach degrades:

In [ ]:
large_motion = (4.0, 3.0)
shift_large = np.float32([[1, 0, large_motion[0]], [0, 1, large_motion[1]]])
frame2_large = cv2.warpAffine(frame1, shift_large, (200, 200))

It_large = frame2_large.astype(np.float64) - frame1.astype(np.float64)

def lucas_kanade_large(x, y, half_win=7):
    x, y = int(round(x)), int(round(y))
    ix = Ix[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    iy = Iy[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    it = It_large[y - half_win:y + half_win + 1, x - half_win:x + half_win + 1].ravel()
    A = np.stack([ix, iy], axis=1)
    solution, *_ = np.linalg.lstsq(A, -it, rcond=None)
    return solution

flows_large_manual = np.array([lucas_kanade_large(p[0][0], p[0][1]) for p in corners])

next_pts, status, _ = cv2.calcOpticalFlowPyrLK(frame1, frame2_large, corners, None, winSize=(15, 15), maxLevel=2)
cv_flow = (next_pts - corners).reshape(-1, 2)[status.ravel() == 1]

print(f'true motion:                          {large_motion}')
print(f'single-shot manual LK estimate:       ({flows_large_manual[:, 0].mean():.3f}, {flows_large_manual[:, 1].mean():.3f})  <- degraded')
print(f'cv2.calcOpticalFlowPyrLK estimate:    ({cv_flow[:, 0].mean():.3f}, {cv_flow[:, 1].mean():.3f})  <- accurate')

OpenCV's `calcOpticalFlowPyrLK` handles large motions by running Lucas-Kanade **iteratively** (re-warping and re-linearizing until convergence) on an **image pyramid** (Lesson 11) &mdash; estimate coarsely on a small, blurry version of the image first, then refine level by level. This combination lets it recover large motions accurately even though the underlying linear approximation is only valid locally, one small step at a time.

### Visualizing sparse flow vectors

In [ ]:
vis = cv2.cvtColor(frame1, cv2.COLOR_GRAY2BGR)
for (p0,), (p1,), ok in zip(corners, next_pts, status.ravel()):
    if not ok:
        continue
    cv2.arrowedLine(vis, tuple(p0.astype(int)), tuple(p1.astype(int)), (0, 0, 255), 1, tipLength=0.3)
    cv2.circle(vis, tuple(p0.astype(int)), 2, (0, 255, 0), -1)

plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f'Tracked corners, true motion = {large_motion}')
plt.axis('off')
plt.show()

## Dense optical flow: Farneback's method

Lucas-Kanade (as used above) is **sparse**: it only estimates flow at a chosen set of good points. `cv2.calcOpticalFlowFarneback` instead estimates a flow vector at *every* pixel, by locally approximating each neighborhood with a polynomial and comparing the polynomial expansions between frames.

In [ ]:
dense_motion = (5.0, -3.0)
shift_dense = np.float32([[1, 0, dense_motion[0]], [0, 1, dense_motion[1]]])
frame2_dense = cv2.warpAffine(frame1, shift_dense, (200, 200))

flow = cv2.calcOpticalFlowFarneback(frame1, frame2_dense, None, pyr_scale=0.5, levels=3,
                                     winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)

print(f'true motion: {dense_motion}')
print(f'mean flow, ALL pixels:      ({flow[..., 0].mean():.2f}, {flow[..., 1].mean():.2f})  <- biased low')

gradient_mag = np.hypot(cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3),
                         cv2.Sobel(frame1.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3))
textured = gradient_mag > 50
textured[:15, :] = textured[-15:, :] = textured[:, :15] = textured[:, -15:] = False  # avoid warp border artifacts

print(f'mean flow, TEXTURED pixels: ({flow[textured, 0].mean():.2f}, {flow[textured, 1].mean():.2f})  <- accurate')

This is the aperture problem again, at its most extreme: over flat, textureless background, there's no local information at all to estimate motion from, so the flow there is unreliable and drags the whole-image average away from the true value. Restricting to textured (high-gradient) pixels recovers the true motion almost exactly.

In [ ]:
def flow_to_color(flow):
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros(flow.shape[:2] + (3,), dtype=np.uint8)
    hsv[..., 0] = angle * 180 / np.pi / 2       # hue = direction
    hsv[..., 1] = 255                            # full saturation
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)  # value = speed
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(frame1, cmap='gray')
axes[0].set_title('Frame 1')
axes[1].imshow(flow_to_color(flow))
axes[1].set_title(f'Dense flow field\n(color = direction, brightness = speed)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Every moving circle produces the *same* color (since they all share the same true motion here) &mdash; in a real video with independently moving objects, this color-coding immediately separates different motions at a glance, which is exactly why it's the standard way to visualize dense flow fields.

### Exercise

1. Rerun the sparse Lucas-Kanade comparison with `large_motion = (10.0, 8.0)`. Does `cv2.calcOpticalFlowPyrLK` still recover it accurately? At what point (try increasingly large motions) does it start to fail, and why would you expect a pyramid to help push that limit further out?
2. Modify the synthetic scene so the circles move with *different* velocities (e.g. half moving one way, half another). Re-run the dense Farneback color visualization and confirm the two motions appear as two distinct colors.
3. In the aperture-problem demo, construct a small textured (non-edge) patch instead of a straight edge, and show that *both* components of an arbitrary motion vector are recoverable from it &mdash; unlike the edge case, this should not have an invisible direction of motion.